# QC-CNN-Parallel: Kaggle T4 Experiment Runner

**Paper:** *A Parallel Hybrid Quantum-Classical Convolutional Design Using Parameterized Quantum Circuits for Image Classification*  
**Quantum Engineering (2026), article 6643049**

This notebook:
1. Clones the private GitHub repo (`Pawangandhi15/QC-CNN-Parallel`)
2. Installs all dependencies
3. Verifies T4 GPU availability
4. Runs Smoke Test → Experiment 1 → Experiment 2 → Experiment 3
5. Saves results and plots to `/kaggle/working/results/`

> **Runtime:** Set accelerator to **GPU T4 x1** in Settings before running.

## 1. GitHub — Clone Repository

In [ ]:
import os

# ── GitHub credentials ──────────────────────────────────────────────────────
GITHUB_TOKEN = "YOUR_GITHUB_PAT_HERE"
REPO_OWNER   = "Pawangandhi15"
REPO_NAME    = "QC-CNN-Parallel"
REPO_BRANCH  = "main"

CLONE_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
REPO_DIR  = f"/kaggle/working/{REPO_NAME}"

if os.path.exists(REPO_DIR):
    print("Repo already cloned — pulling latest changes...")
    os.system(f"git -C {REPO_DIR} pull")
else:
    print(f"Cloning {REPO_OWNER}/{REPO_NAME} ...")
    ret = os.system(f"git clone --branch {REPO_BRANCH} --depth 1 {CLONE_URL} {REPO_DIR}")
    if ret != 0:
        raise RuntimeError("git clone failed — check GITHUB_TOKEN and repo name.")

print("✓ Repository ready at:", REPO_DIR)
os.listdir(REPO_DIR)

## 2. Install Dependencies

In [ ]:
import subprocess, sys

req_file = os.path.join(REPO_DIR, "implementation", "requirements.txt")
print("Installing from:", req_file)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", req_file],
    check=True
)

# Verify key imports
import torch, pennylane, torchvision, sklearn, matplotlib
print(f"\n✓ torch      {torch.__version__}")
print(f"✓ pennylane  {pennylane.__version__}")
print(f"✓ torchvision {torchvision.__version__}")
print(f"✓ sklearn    {sklearn.__version__}")
print(f"✓ matplotlib {matplotlib.__version__}")

## 3. GPU / Environment Check

In [ ]:
import torch

print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU            : {gpu.name}")
    print(f"VRAM           : {gpu.total_memory / 1e9:.1f} GB")
    DEVICE = torch.device("cuda")
else:
    print("WARNING: No GPU found — running on CPU (much slower).")
    DEVICE = torch.device("cpu")

print("\nDevice used    :", DEVICE)

## 4. Path Setup

In [ ]:
import sys, os
from pathlib import Path

IMPL_DIR    = os.path.join(REPO_DIR, "implementation")
WORKING_DIR = "/kaggle/working"

# Insert at front so local modules take precedence
if IMPL_DIR not in sys.path:
    sys.path.insert(0, IMPL_DIR)

# Change CWD so relative paths (data/, results/) resolve inside /kaggle/working
os.chdir(WORKING_DIR)
Path("results").mkdir(exist_ok=True)

print("IMPL_DIR :", IMPL_DIR)
print("CWD      :", os.getcwd())

# Quick import sanity-check
from models import QCCNNParallel, ClassicalCNN
print("\n✓ models imported successfully")

## 5. Smoke Test — Verify Forward/Backward Pass

In [ ]:
import torch
from models import QCCNNParallel, ClassicalCNN
from models.quantum_circuit import make_noisy_circuit

print("=" * 60)
print("  SMOKE TEST")
print("=" * 60)

B       = 2
images  = torch.rand(B, 1, 28, 28)
labels  = torch.randint(0, 10, (B,))
loss_fn = torch.nn.CrossEntropyLoss()

for name, model in [("QCCNNParallel", QCCNNParallel(10)),
                    ("ClassicalCNN",  ClassicalCNN(10))]:
    model = model.cpu()
    opt   = torch.optim.Adam(model.parameters(), lr=0.01)
    opt.zero_grad()
    logits = model(images)
    loss   = loss_fn(logits, labels)
    loss.backward()
    opt.step()
    params = sum(p.numel() for p in model.parameters())
    print(f"  ✓ {name:<20}  loss={loss.item():.4f}  params={params:,}")

print("\n  Testing noisy circuits (default.mixed) ...")
for nt in ["bit_flip", "phase_flip", "depolarizing"]:
    qnode  = make_noisy_circuit(nt, 0.1)
    model  = QCCNNParallel(10, qnode=qnode)
    logits = model(images[:1])
    loss   = loss_fn(logits, labels[:1])
    print(f"  ✓ Noisy ({nt:<14})  loss={loss.item():.4f}")

print("\n  ✓ All smoke tests passed!")

## 6. Experiment 1 — PQC Circuit Selection Study (Tables 2 & 3)

Evaluates 11 candidate PQC architectures on expressibility, entangling capability, and classification accuracy on MNIST / Fashion-MNIST.

In [ ]:
import sys
sys.path.insert(0, IMPL_DIR)

from experiments.experiment1_circuit_selection import run_experiment1

print("=" * 60)
print("  EXPERIMENT 1: PQC Selection Study")
print("=" * 60)

# n_sims=500 for a faster Kaggle run; paper uses 5000.
# Set n_sims=5000 and run_metrics=True for full reproduction.
exp1_results = run_experiment1(run_metrics=True, n_sims=500)
print("\n✓ Experiment 1 complete.")

## 7. Experiment 2 — Classification Benchmarks (Figures 6–8, Table 4)

Trains QC-CNN-Parallel vs Classical CNN on MNIST, Fashion-MNIST, and Overhead-MNIST.

In [ ]:
from experiments.experiment2_classification import run_experiment2

print("=" * 60)
print("  EXPERIMENT 2: Classification Benchmarks")
print("=" * 60)

# Run on all three paper datasets; Overhead-MNIST will be skipped
# if the dataset file is not present in /kaggle/working/data/
exp2_results = run_experiment2(
    datasets_to_run=["mnist", "fashion_mnist", "overhead_mnist"]
)
print("\n✓ Experiment 2 complete.")

## 8. Experiment 3 — Noise Robustness (Tables 5–8)

Evaluates QC-CNN-Parallel under data noise (Gaussian) and quantum noise channels (bit-flip, phase-flip, depolarizing) at error rates p ∈ {0.1, 0.2, 0.3}.

In [ ]:
from experiments.experiment3_noise_robustness import run_experiment3

print("=" * 60)
print("  EXPERIMENT 3: Noise Robustness")
print("=" * 60)

# max_test_samples=500 speeds up the Kaggle run.
# Remove the argument (or set None) for full 10,000 sample evaluation.
exp3_results = run_experiment3(max_test_samples=500)
print("\n✓ Experiment 3 complete.")

## 9. Display Results & Plots

In [ ]:
import json, glob
from pathlib import Path
from IPython.display import Image, display

RESULTS_ROOT = Path("/kaggle/working/results")

# ── Print JSON summaries ────────────────────────────────────────────────────
for jf in sorted(RESULTS_ROOT.rglob("*.json")):
    print(f"\n{'='*60}")
    print(f"  {jf.relative_to(RESULTS_ROOT)}")
    print(f"{'='*60}")
    with open(jf) as f:
        print(json.dumps(json.load(f), indent=2))

# ── Display PNG plots ───────────────────────────────────────────────────────
print("\n" + "="*60)
print("  GENERATED PLOTS")
print("="*60)
for img in sorted(RESULTS_ROOT.rglob("*.png")):
    print(f"\n▶ {img.name}")
    display(Image(filename=str(img)))

## 10. Sync Results Back to GitHub

In [ ]:
import subprocess, shutil, os
from pathlib import Path

# Copy Kaggle results into the cloned repo
SRC = Path("/kaggle/working/results")
DST = Path(REPO_DIR) / "results" / "kaggle_run"
DST.mkdir(parents=True, exist_ok=True)

for f in SRC.rglob("*"):
    if f.is_file():
        rel   = f.relative_to(SRC)
        dest  = DST / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dest)

print(f"✓ Copied results to {DST}")

def git(*args, cwd=REPO_DIR):
    result = subprocess.run(["git"] + list(args), cwd=cwd,
                            capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    print(result.stdout)
    return result.returncode

git("config", "user.email", "kaggle-bot@qc-cnn.run")
git("config", "user.name",  "Kaggle T4 Runner")
git("add", str(DST))
git("commit", "-m", "chore: add Kaggle T4 experiment results")
ret = git("push", "origin", REPO_BRANCH)

if ret == 0:
    print("\n✓ Results pushed to GitHub successfully!")
else:
    print("\n⚠ Push failed — check token permissions or branch protection rules.")